# GLP1 EDA

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/chembl_glp1.csv", sep=';')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

C:\Users\Tom\AppData\Local\Temp\ipykernel_6896\2747045884.py:4: DtypeWarning: Columns (0: Assay Subcellular Fraction) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/chembl_glp1.csv", sep=';')


* Do the datatypes make sense?


In [2]:

print(df.head())
print(df.info())
print(df.isna().sum())
print(f"There are {df.isna().sum()[df.isna().sum() > (0.9 * 113886)].count()} columns with >90% null rows and {df.isna().sum()[df.isna().sum() == 0].count()} with no null rows")

  Molecule ChEMBL ID Molecule Name  Molecule Max Phase  Molecular Weight  #RO5 Violations  AlogP  Compound Key                                             Smiles Standard Type Standard Relation  Standard Value Standard Units  pChEMBL Value Data Validity Comment                Comment    Uo Units  Ligand Efficiency BEI  Ligand Efficiency LE  Ligand Efficiency LLE  Ligand Efficiency SEI  Potential Duplicate Assay ChEMBL ID                                  Assay Description Assay Type BAO Format ID          BAO Label Assay Organism  Assay Tissue ChEMBL ID  Assay Tissue Name Assay Cell Type Assay Subcellular Fraction  Assay Parameters  Assay Variant Accession  Assay Variant Mutation Target ChEMBL ID                       Target Name Target Organism     Target Type Document ChEMBL ID  Source ID     Source Description Document Journal  Document Year Cell ChEMBL ID Properties Action Type Standard Text Value   Value
0      CHEMBL4095693           NaN                 NaN            518.80      

* How many missing values are there in each column?
* How many distinct values does each column have?

In [3]:
print(f"There are {df.isna().sum()[df.isna().sum() > (0.9 * 113886)].count()} columns with >90% null rows and {df.isna().sum()[df.isna().sum() == 0].count()} with no null rows \n")
summary = pd.DataFrame()
summary['null_values'] = df.isna().sum()
summary['unique_values'] = df.nunique()
print(f"{summary} \n")
print(f"The max unique values in a column is {summary['unique_values'].max()}, which is {df['Molecule ChEMBL ID'].count() - summary['unique_values'].max()} short of the {df['Molecule ChEMBL ID'].count()} rows")
print(f" There are {df.duplicated().sum()} duplicated rows")

There are 22 columns with >90% null rows and 16 with no null rows 

                            null_values  unique_values
Molecule ChEMBL ID                    0         107368
Molecule Name                    112271            804
Molecule Max Phase               112630              6
Molecular Weight                     49          19248
#RO5 Violations                    1968              5
AlogP                              1968           1072
Compound Key                          0         107088
Smiles                               52         107346
Standard Type                         0             25
Standard Relation                107696              6
Standard Value                     1223           2657
Standard Units                      831             17
pChEMBL Value                    111574            542
Data Validity Comment            113680              1
Comment                            2796            800
Uo Units                           1428             

* Which compounds have more than one row? How many do they have?

In [4]:
print(df['Molecule ChEMBL ID'].value_counts())
mol_count = df['Molecule ChEMBL ID'].value_counts()
mask = df['Molecule ChEMBL ID'].value_counts() > 1

print(f"There are {mol_count[mask].shape[0]} compounds with more than 1 row, and they have up to {mol_count[mask].max()} rows")

Molecule ChEMBL ID
CHEMBL4518483    59
CHEMBL410972     55
CHEMBL414357     33
CHEMBL4084119    29
CHEMBL5186808    23
                 ..
CHEMBL5314631     1
CHEMBL6161510     1
CHEMBL6169222     1
CHEMBL6168355     1
CHEMBL6171075     1
Name: count, Length: 107368, dtype: int64
There are 3811 compounds with more than 1 row, and they have up to 59 rows


* How does CHEMBL4518483 have 59 rows if there are only 17 different Standard Units? 

In [5]:
# print(df[df['Molecule ChEMBL ID'] == 'CHEMBL4518483'])

selection = df[df['Molecule ChEMBL ID'] == 'CHEMBL4518483']

print(f"CHEMBL4518483 has been tested in {selection['Assay ChEMBL ID'].nunique()} unique assays, each with up to {selection.groupby('Assay ChEMBL ID')['Molecule ChEMBL ID'].agg('count').max()} values")

CHEMBL4518483 has been tested in 49 unique assays, each with up to 6 values


* How many different standard units are there and what are they?

In [6]:
grouped_std_units = df.groupby('Standard Units')['Molecule ChEMBL ID'].agg('count')
print(grouped_std_units)
print(f"\nThere are {grouped_std_units.count()} different units, but nM covers {100  * grouped_std_units['nM']/grouped_std_units.sum():.1f}% of the entries")
print(df['Standard Units'].value_counts(normalize=True))

Standard Units
%               2115
10'8pM             1
10^-1/s            1
10^-2/s           11
10^-3/s            4
10^-5M             1
10^-6M             3
10^2/Ms            1
10^3(1/Ms)         3
10^3/M.s           2
10^3/M/s           2
10^4/Ms            7
10^5/M.s           1
hr                 1
nM            110342
pmol/L             8
s-1              552
Name: Molecule ChEMBL ID, dtype: int64

There are 17 different units, but nM covers 97.6% of the entries
Standard Units
nM            0.976003
%             0.018708
s-1           0.004883
10^-2/s       0.000097
pmol/L        0.000071
10^4/Ms       0.000062
10^-3/s       0.000035
10^-6M        0.000027
10^3(1/Ms)    0.000027
10^3/M/s      0.000018
10^3/M.s      0.000018
hr            0.000009
10^2/Ms       0.000009
10'8pM        0.000009
10^-5M        0.000009
10^-1/s       0.000009
10^5/M.s      0.000009
Name: proportion, dtype: float64


* Why does assay organism contain nulls but only 1 unique value? Confirm it does include nulls

In [7]:
assay_organism = df[df['Assay Organism'] == "Homo sapiens"]
print(assay_organism)

assay_organism_nan = df[df['Assay Organism'].isna()]
print(assay_organism_nan)

       Molecule ChEMBL ID Molecule Name  Molecule Max Phase  Molecular Weight  #RO5 Violations  AlogP  Compound Key                                             Smiles Standard Type Standard Relation  Standard Value Standard Units  pChEMBL Value  Data Validity Comment                Comment    Uo Units  Ligand Efficiency BEI  Ligand Efficiency LE  Ligand Efficiency LLE  Ligand Efficiency SEI  Potential Duplicate Assay ChEMBL ID                                  Assay Description Assay Type BAO Format ID          BAO Label Assay Organism  Assay Tissue ChEMBL ID  Assay Tissue Name Assay Cell Type Assay Subcellular Fraction  Assay Parameters  Assay Variant Accession  Assay Variant Mutation Target ChEMBL ID                       Target Name Target Organism     Target Type Document ChEMBL ID  Source ID     Source Description Document Journal  Document Year Cell ChEMBL ID                       Properties Action Type Standard Text Value    Value
0           CHEMBL4095693           NaN          

What are the 25 unique values for Standard Type?

In [8]:
print(df['Standard Type'].value_counts())

Standard Type
Potency                  107577
EC50                       2365
%Inhib (Mean)              1032
%Max (Mean)                 558
kon                         552
k_off                       552
IC50                        350
Emax                        300
Activity                    274
Ki                           63
Ratio IC50                   55
Ratio EC50                   53
FC                           49
Kd                           20
Inhibition                   19
Efficacy                     19
Ka                           16
Emin                         10
Kdiss                         9
Ratio                         5
Ke                            4
T1/2                          1
Mean fold stimulation         1
RLU                           1
pEC50                         1
Name: count, dtype: int64


Filtering to 'Potency' type measurements:

In [16]:
filtered = df[df['Standard Type'] == 'Potency']
print(filtered['Standard Units'].value_counts(), '\n')

print(filtered.columns[filtered.isna().all()]) # Which columns are null only
filtered_2 = filtered.dropna(axis=1, how='all') # Drop null columns
filtered_2.info()

Standard Units
nM    107577
Name: count, dtype: int64 

Index(['Standard Relation', 'pChEMBL Value', 'Data Validity Comment', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties', 'Action Type', 'Standard Text Value'], dtype='str')
<class 'pandas.DataFrame'>
Index: 107577 entries, 45 to 113187
Data columns (total 28 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Molecule ChEMBL ID   107577 non-null  str    
 1   Molecule Name        640 non-null     str    
 2   Molecule Max Phase   394 non-null     float64
 3   Molecular Weight     107562 non-null  float64
 4   #RO5 Violations      107497 non-null  float64
 5   AlogP                107497

Which assays are contained in the data:

In [20]:
print(filtered_2['Assay Description'].value_counts(), '\n')
print(filtered_2['Assay ChEMBL ID'].value_counts())

filtered_3 = filtered_2[filtered_2['Assay ChEMBL ID'] == 'CHEMBL2114788']

Assay Description
PubChem BioAssay. qHTS of GLP-1 Receptor Inverse Agonists (Inhibition Mode). (Class of assay: confirmatory)     103901
PubChem BioAssay. qHTS of GLP-1 Receptor Agonists. (Class of assay: confirmatory)                                 3541
PubChem BioAssay. qHTS of GLP-1 Receptor Agonists: Hit Validation.   (Class of assay: confirmatory)                135
Name: count, dtype: int64 

Assay ChEMBL ID
CHEMBL2114788    103901
CHEMBL2114931      3541
CHEMBL3215152       135
Name: count, dtype: int64
